# 02 — Behavioural metrics

These metrics are derived purely from the translation run — they need no graph database and no reference query. They tell you how the translator *behaved*, not how *correct* its output was:

| Metric | What it measures | Why it matters |
|---|---|---|
| **Pass@1** | Fraction of translations that validated on the first generation, before any fix iteration. | Tells you how often the LLM gets it right zero-shot. Standard HumanEval-style metric. |
| **Pass@k** | Fraction of translations that validated within `k` iterations (here `k = max_iterations = 3`). | Headline 'does the fix loop ever close?' number. |
| **Validation success rate** | Same as Pass@k for our config — kept separately because some literature distinguishes them. | Sanity-check cross-reference. |
| **Mean iterations used** | Average value of `iterations_used` over all records. | Measures how much work the fix loop actually does. Closer to 1.0 = mostly first-shot successes. |
| **Mean duration** | Wall-clock time per translation (LLM latency + validator latency). | A practicality metric — useful for thesis 'cost of deployment' discussion. |
| **Total input/output tokens** | Sum of token usage across all calls. | Direct input to the cost calculation. |
| **Cost per success** | `(total_cost_usd) / (successful_translations)`. | The economics of using this stack — answers 'how much per usable query?'. |

All numbers are computed both overall and stratified by `schema_name × dialect`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

OUTPUTS_DIR = REPO_ROOT / "evaluation" / "outputs"
RECORDS_PATH = OUTPUTS_DIR / "records.json"
OUT_CSV = OUTPUTS_DIR / "metrics_behavioural.csv"

assert RECORDS_PATH.exists(), "Run 01_translation_run.ipynb first."
records = json.loads(RECORDS_PATH.read_text())
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records from {RECORDS_PATH}")

## Pricing constants

Costs are reported in USD. Defaults are Anthropic's posted prices for Claude Opus (per million tokens). Adjust if your eval uses Sonnet or Haiku — or if Anthropic's pricing has moved since this notebook was written.

Source: https://docs.claude.com/en/docs/about-claude/pricing

In [ ]:
# USD per million tokens. Updated 2026-06.
PRICING = {
    "input_per_mtok": 15.0,
    "output_per_mtok": 75.0,
}


def usd_cost(input_tokens: int, output_tokens: int) -> float:
    return (input_tokens / 1e6) * PRICING["input_per_mtok"] + (output_tokens / 1e6) * PRICING["output_per_mtok"]

## Per-record signals

Each behavioural signal we care about is already a column on `records.json`. We just need to derive `pass@1` (validated AND iterations_used == 1) as a flag and add a per-record cost.

In [ ]:
df["pass_at_1"] = df["validation_passed"] & (df["iterations_used"] == 1)
df["cost_usd"] = df.apply(lambda r: usd_cost(r["input_tokens"], r["output_tokens"]), axis=1)
df[["query_id", "dialect", "iterations_used", "validation_passed", "pass_at_1", "input_tokens", "output_tokens", "cost_usd"]].head(10)

## Overall numbers

In [ ]:
MAX_ITERATIONS = int(df["iterations_used"].max())  # snapshot the actual cap that was used


def aggregate(group: pd.DataFrame) -> pd.Series:
    successes = group["validation_passed"].sum()
    return pd.Series(
        {
            "n": len(group),
            "pass@1": group["pass_at_1"].mean(),
            f"pass@{MAX_ITERATIONS}": group["validation_passed"].mean(),
            "mean_iterations": group["iterations_used"].mean(),
            "mean_duration_s": group["duration_seconds"].mean(),
            "total_input_tok": int(group["input_tokens"].sum()),
            "total_output_tok": int(group["output_tokens"].sum()),
            "total_cost_usd": group["cost_usd"].sum(),
            "cost_per_success_usd": group["cost_usd"].sum() / successes if successes else float("nan"),
        }
    )


overall = aggregate(df).to_frame(name="overall").T
overall

## Stratified: schema × dialect

TPC-H and LDBC have different difficulty distributions and different graph schemas — surface each slice separately so we can spot dialect-specific failure modes.

In [ ]:
by_schema_dialect = df.groupby(["schema_name", "dialect"], group_keys=False).apply(aggregate)
by_schema_dialect

## Stratified: difficulty

Pass@k stratified by difficulty is one of the most diagnostic plots for the thesis — it tells the reader 'easy queries work, hard ones don't yet'.

In [ ]:
by_difficulty = df.groupby(["difficulty"], group_keys=False).apply(aggregate)
by_difficulty.reindex(["easy", "medium", "hard"])

## Save per-record CSV

Downstream notebooks (and the final report) read `metrics_behavioural.csv`. Per-record granularity lets the report notebook join behavioural metrics with structural / distance / execution metrics on `(query_id, dialect)`.

In [ ]:
out = df[
    [
        "query_id", "schema_name", "dialect", "difficulty", "model_name",
        "validation_passed", "pass_at_1", "iterations_used", "duration_seconds",
        "input_tokens", "output_tokens", "cost_usd",
    ]
].copy()
out.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(out)} rows to {OUT_CSV}")